# Assignment 1 — myPOS 3.0 POS Tagging with CRF

Linn Thant -AI Engineering (Fundamental), Batch-2  


1. Library install လုပ်ပြီး import ပြုလုပ်ခြင်း


In [1]:
%pip install -q sklearn-crfsuite scikit-learn tabulate

import random
import re
import time
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import sklearn_crfsuite
from tabulate import tabulate

DATA = Path("data")
DATA.mkdir(exist_ok=True)
SEED = 42
print("ready — same CRFsuite backend as the class tutorial")


Note: you may need to restart the kernel to use updated packages.
ready — same CRFsuite backend as the class tutorial


2. myPOS corpus နဲ့ `otest.1k` ဖိုင် download ပြုလုပ်ခြင်း


In [2]:
CORPUS_URL = (
    "https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/"
    "refs/heads/master/corpus-ver-3.0/corpus"
)
NEEDED = ["mypos-ver.3.0.shuf.nopipe.txt", "otest.1k.nopipe.txt"]

for fname in NEEDED:
    dest = DATA / fname
    if not dest.exists():
        print("fetching", fname)
        urllib.request.urlretrieve(f"{CORPUS_URL}/{fname}", dest)
    print(f"{fname:40s} {dest.stat().st_size:10,d} bytes")


mypos-ver.3.0.shuf.nopipe.txt             9,581,544 bytes
otest.1k.nopipe.txt                         229,758 bytes


3. ဖိုင်တွေကို `word/tag` စာကြောင်းတွေအဖြစ် parse ပြုလုပ်ခြင်း


In [3]:
def read_word_tag(path):
    """Each token is word/tag. Split from the RIGHT (dates may contain '/')."""
    sents, bad = [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            sent = []
            for tok in line.split():
                w, sep, t = tok.rpartition("/")
                if not sep:
                    bad += 1
                    continue
                sent.append((w, t))
            if sent:
                sents.append(sent)
    return sents, bad


all_sents, bad_all = read_word_tag(DATA / "mypos-ver.3.0.shuf.nopipe.txt")
gold_test, bad_test = read_word_tag(DATA / "otest.1k.nopipe.txt")
label_set = sorted({t for s in all_sents for _, t in s})

print(f"corpus : {len(all_sents):,} sents / {sum(map(len, all_sents)):,} tokens  (bad={bad_all})")
print(f"otest  : {len(gold_test):,} sents / {sum(map(len, gold_test)):,} tokens  (bad={bad_test})")
print("labels :", " ".join(label_set))
print("sample :", " ".join(f"{w}/{t}" for w, t in gold_test[0]))


corpus : 43,196 sents / 564,517 tokens  (bad=0)
otest  : 1,000 sents / 13,468 tokens  (bad=0)
labels : abb adj adv conj fw int n num part ppm pron punc sb tn v
sample : တစ်/tn ကိုက်/n ကို/ppm ဝမ်/n ခုနှစ်ထောင်/tn ပါ/part ။/punc


4. leakage မဖြစ်အောင် train / dev / test ခွဲခြင်း


In [4]:
def joined(sent):
    return " ".join(w for w, _ in sent)


in_full = sum(1 for s in gold_test if joined(s) in {joined(x) for x in all_sents})
print(f"otest sentences found inside full corpus: {in_full}/{len(gold_test)}")

# remove one exact copy of each test sentence
bag = Counter(tuple(s) for s in gold_test)
pool = []
for s in all_sents:
    key = tuple(s)
    if bag[key] > 0:
        bag[key] -= 1
    else:
        pool.append(s)

rng = random.Random(SEED)
rng.shuffle(pool)
DEV_N = 2000
dev_sents = pool[:DEV_N]
train_sents = pool[DEV_N:]

print(f"pool={len(pool):,}  train={len(train_sents):,}  "
      f"dev={len(dev_sents):,}  test={len(gold_test):,}")
assert len(pool) == 42196, f"expected 42196 train pool, got {len(pool)}"


otest sentences found inside full corpus: 1000/1000
pool=42,196  train=40,196  dev=2,000  test=1,000


5. tutorial baseline feature template ပြင်ဆင်ခြင်း


In [5]:
def tutorial_features(sent, i):
    """Exact idea of class-5 chunking.py templates + BOS/EOS."""
    ws = [w for w, _ in sent]
    n, feats = len(ws), {}
    for d in (-2, -1, 0, 1, 2):
        j = i + d
        if 0 <= j < n:
            feats[f"w[{d}]"] = ws[j]
    if i > 0:
        feats["w[-1]|w[0]"] = ws[i - 1] + "|" + ws[i]
    if i + 1 < n:
        feats["w[0]|w[1]"] = ws[i] + "|" + ws[i + 1]
    if i == 0:
        feats["BOS"] = True
    if i == n - 1:
        feats["EOS"] = True
    return feats


def make_xy(sents, feat_fn):
    X = [[feat_fn(s, i) for i in range(len(s))] for s in sents]
    y = [[t for _, t in s] for s in sents]
    return X, y


print("demo features @ position 0:")
for k, v in tutorial_features(gold_test[0], 0).items():
    print(f"  {k} = {v}")


demo features @ position 0:
  w[0] = တစ်
  w[1] = ကိုက်
  w[2] = ကို
  w[0]|w[1] = တစ်|ကိုက်
  BOS = True


6. evaluation function နဲ့ CRF train helper ပြင်ဆင်ခြင်း


In [6]:
def score(y_true, y_pred):
    hit = defaultdict(int)
    pred_n = defaultdict(int)
    gold_n = defaultdict(int)
    ok_tok = n_tok = ok_sent = n_sent = 0
    for gseq, hseq in zip(y_true, y_pred):
        n_sent += 1
        ok_sent += (gseq == hseq)
        for g, h in zip(gseq, hseq):
            n_tok += 1
            gold_n[g] += 1
            pred_n[h] += 1
            if g == h:
                ok_tok += 1
                hit[g] += 1
    rows = []
    for lab in sorted(set(gold_n) | set(pred_n), key=lambda x: -gold_n[x]):
        p = hit[lab] / pred_n[lab] if pred_n[lab] else 0.0
        r = hit[lab] / gold_n[lab] if gold_n[lab] else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        rows.append((lab, hit[lab], pred_n[lab], gold_n[lab], p, r, f))
    macro = tuple(sum(r[i] for r in rows) / len(rows) for i in (4, 5, 6))
    return {
        "rows": rows, "macro": macro,
        "item": ok_tok / n_tok, "inst": ok_sent / n_sent,
        "tok": (ok_tok, n_tok), "sent": (ok_sent, n_sent),
    }


def show(title, res):
    print(title)
    print(tabulate(
        [(a, m, mo, rf, f"{p:.4f}", f"{r:.4f}", f"{f:.4f}")
         for a, m, mo, rf, p, r, f in res["rows"]],
        headers=["label", "#match", "#model", "#ref", "P", "R", "F1"],
    ))
    print(f"Macro-average precision, recall, F1: "
          f"({res['macro'][0]:.6f}, {res['macro'][1]:.6f}, {res['macro'][2]:.6f})")
    print(f"Item accuracy: {res['tok'][0]} / {res['tok'][1]} ({res['item']:.4f})")
    print(f"Instance accuracy: {res['sent'][0]} / {res['sent'][1]} ({res['inst']:.4f})")


def fit_crf(X, y, c1=0.1, c2=0.01, iters=100):
    # mirrors: crfsuite learn (L-BFGS, L1/L2)
    model = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=c1, c2=c2,
        max_iterations=iters, all_possible_transitions=True,
    )
    t0 = time.perf_counter()
    model.fit(X, y)
    print(f"crf learned in {time.perf_counter()-t0:.1f}s  [c1={c1}, c2={c2}, iter={iters}]")
    return model


7. baseline model ကို train လုပ်ပြီး dev set တိုင်းတာခြင်း


In [7]:
X_tr, y_tr = make_xy(train_sents, tutorial_features)
X_dv, y_dv = make_xy(dev_sents, tutorial_features)

base_model = fit_crf(X_tr, y_tr, c1=0.1, c2=0.01, iters=100)
base_dev = score(y_dv, base_model.predict(X_dv))
show("=== tutorial template · DEV (like crfsuite tag -qt) ===", base_dev)


crf learned in 54.9s  [c1=0.1, c2=0.01, iter=100]
=== tutorial template · DEV (like crfsuite tag -qt) ===
label      #match    #model    #ref       P       R      F1
-------  --------  --------  ------  ------  ------  ------
part         6059      6222    6231  0.9738  0.9724  0.9731
n            5598      5949    5730  0.941   0.977   0.9586
ppm          3966      4042    4022  0.9812  0.9861  0.9836
v            3669      3836    3878  0.9565  0.9461  0.9513
punc         2487      2488    2488  0.9996  0.9996  0.9996
pron          881       902     914  0.9767  0.9639  0.9703
conj          736       804     798  0.9154  0.9223  0.9189
adj           625       732     754  0.8538  0.8289  0.8412
adv           385       415     495  0.9277  0.7778  0.8462
tn            255       262     262  0.9733  0.9733  0.9733
num           226       234     242  0.9658  0.9339  0.9496
fw             83        86     151  0.9651  0.5497  0.7004
int            28        29      33  0.9655  0.8485  0

8. script / affix feature တွေ ထပ်ထည့်ခြင်း


In [8]:
_MM_LETTER = re.compile(r"[\u1000-\u109F]")
_ASCII_LATIN = re.compile(r"[A-Za-z]")
_ASCII_DIGIT = re.compile(r"[0-9]")
_MM_DIGIT = re.compile(r"[၀-၉]")
_NUM = re.compile(r"^[0-9၀-၉][0-9၀-၉,./\-]*$")
_PUNCT = frozenset("၊။၌၍၏၎!\"#$%&'()*+,./:;<=>?@[\\]^_`{|}~‘’“”-−–—_%°")


def script_of(w: str) -> str:
    if not w:
        return "empty"
    if all(ch in _PUNCT for ch in w):
        return "punct"
    if _NUM.match(w):
        return "number"
    has_lat = bool(_ASCII_LATIN.search(w))
    has_dig = bool(_ASCII_DIGIT.search(w) or _MM_DIGIT.search(w))
    has_mm = bool(_MM_LETTER.search(w))
    if has_lat and not has_mm:
        return "latin_digit" if has_dig else "latin"
    if has_dig and has_mm:
        return "mm_digit_mix"
    if has_dig:
        return "has_digit"
    if has_mm:
        return "myanmar"
    return "other"


def improved_features(sent, i):
    """Tutorial template + script/affix features (my addition)."""
    feats = dict(tutorial_features(sent, i))
    ws = [w for w, _ in sent]
    w, n = ws[i], len(ws)

    feats["script"] = script_of(w)
    feats["len"] = str(min(len(w), 8))
    for k in (1, 2, 3):
        if len(w) > k:
            feats[f"pref{k}"] = w[:k]
            feats[f"suf{k}"] = w[-k:]
    # neighbour scripts help disambiguate particle vs noun contexts
    if i > 0:
        feats["script[-1]"] = script_of(ws[i - 1])
        if len(ws[i - 1]) >= 2:
            feats["suf2[-1]"] = ws[i - 1][-2:]
    if i + 1 < n:
        feats["script[+1]"] = script_of(ws[i + 1])
    return feats


for sample in ["မြန်မာနိုင်ငံ", "COVID", "AI", "၂၀၂၆", "2026", "။", "%", "BBC"]:
    print(f"{sample:16s} -> {script_of(sample)}")


မြန်မာနိုင်ငံ    -> myanmar
COVID            -> latin
AI               -> latin
၂၀၂၆             -> number
2026             -> number
။                -> punct
%                -> punct
BBC              -> latin


9. improved model train လုပ်ပြီး baseline နဲ့ပြန်လည်နှိုင်းယှဉ်ခြင်း


In [9]:
X_tr_r, _ = make_xy(train_sents, improved_features)
X_dv_r, _ = make_xy(dev_sents, improved_features)

improved_model = fit_crf(X_tr_r, y_tr, c1=0.1, c2=0.01, iters=100)
improved_dev = score(y_dv, improved_model.predict(X_dv_r))
show("=== tutorial + improved features · DEV ===", improved_dev)

print("\n--- per-label F1 delta (improved - baseline) ---")
b = {r[0]: r[6] for r in base_dev["rows"]}
r = {r[0]: r[6] for r in improved_dev["rows"]}
for lab in sorted(b, key=lambda x: r.get(x, 0) - b[x], reverse=True):
    print(f"{lab:>5}  {b[lab]:.4f} -> {r.get(lab,0):.4f}  ({r.get(lab,0)-b[lab]:+.4f})")
print(f"\nitem   {base_dev['item']:.4f} -> {improved_dev['item']:.4f}")
print(f"macroF {base_dev['macro'][2]:.4f} -> {improved_dev['macro'][2]:.4f}")


crf learned in 82.2s  [c1=0.1, c2=0.01, iter=100]
=== tutorial + improved features · DEV ===
label      #match    #model    #ref       P       R      F1
-------  --------  --------  ------  ------  ------  ------
part         6059      6230    6231  0.9726  0.9724  0.9725
n            5592      5793    5730  0.9653  0.9759  0.9706
ppm          3967      4047    4022  0.9802  0.9863  0.9833
v            3671      3841    3878  0.9557  0.9466  0.9512
punc         2487      2488    2488  0.9996  0.9996  0.9996
pron          886       908     914  0.9758  0.9694  0.9726
conj          734       802     798  0.9152  0.9198  0.9175
adj           643       748     754  0.8596  0.8528  0.8562
adv           402       449     495  0.8953  0.8121  0.8517
tn            259       267     262  0.97    0.9885  0.9792
num           241       244     242  0.9877  0.9959  0.9918
fw            147       150     151  0.98    0.9735  0.9767
int            29        29      33  1       0.8788  0.9355
abb    

10. DEV set ပေါ်မှာ `c1` / `c2` ရှာခြင်း

`c1` / `c2` က overfitting မဖြစ်အောင် ထိန်းတဲ့ regularization တန်ဖိုးနှစ်ခုဖြစ်ပြီး၊ DEV ပေါ်မှာ အကောင်းဆုံးကို ရွေးပြီးမှ final test ပြုလုပ်ခြင်း


In [10]:
trials = []
for c1 in (0.05, 0.1, 0.5):
    for c2 in (0.001, 0.01):
        m = fit_crf(X_tr_r, y_tr, c1=c1, c2=c2, iters=100)
        res = score(y_dv, m.predict(X_dv_r))
        trials.append((c1, c2, res["item"], res["macro"][2], res["inst"], m))
        print(f"  c1={c1:<4} c2={c2:<6}  "
              f"item={res['item']:.4f}  macroF={res['macro'][2]:.4f}  "
              f"sent={res['inst']:.4f}")

# pick best macro-F1, break ties with item accuracy
best = max(trials, key=lambda t: (t[3], t[2]))
C1, C2 = best[0], best[1]
print(f"\nchosen on DEV -> c1={C1}, c2={C2}  "
      f"(macroF={best[3]:.4f}, item={best[2]:.4f})")


crf learned in 96.3s  [c1=0.05, c2=0.001, iter=100]
  c1=0.05 c2=0.001   item=0.9651  macroF=0.9480  sent=0.6810
crf learned in 98.6s  [c1=0.05, c2=0.01, iter=100]
  c1=0.05 c2=0.01    item=0.9656  macroF=0.9514  sent=0.6880
crf learned in 98.5s  [c1=0.1, c2=0.001, iter=100]
  c1=0.1  c2=0.001   item=0.9658  macroF=0.9512  sent=0.6825
crf learned in 97.8s  [c1=0.1, c2=0.01, iter=100]
  c1=0.1  c2=0.01    item=0.9661  macroF=0.9525  sent=0.6875
crf learned in 98.2s  [c1=0.5, c2=0.001, iter=100]
  c1=0.5  c2=0.001   item=0.9669  macroF=0.9525  sent=0.6960
crf learned in 96.7s  [c1=0.5, c2=0.01, iter=100]
  c1=0.5  c2=0.01    item=0.9673  macroF=0.9529  sent=0.7000

chosen on DEV -> c1=0.5, c2=0.01  (macroF=0.9529, item=0.9673)


11. final model နဲ့ `otest.1k` test တိုင်းတာခြင်း


In [11]:
combo = train_sents + dev_sents
X_all, y_all = make_xy(combo, improved_features)
X_te, y_te = make_xy(gold_test, improved_features)

final = fit_crf(X_all, y_all, c1=C1, c2=C2, iters=120)
test_res = score(y_te, final.predict(X_te))
show("=== FINAL · improved features · otest.1k ===", test_res)


crf learned in 120.9s  [c1=0.5, c2=0.01, iter=120]
=== FINAL · improved features · otest.1k ===
label      #match    #model    #ref       P       R      F1
-------  --------  --------  ------  ------  ------  ------
part         3105      3192    3189  0.9727  0.9737  0.9732
n            2930      3034    3000  0.9657  0.9767  0.9712
ppm          2018      2056    2060  0.9815  0.9796  0.9806
v            1911      2001    2010  0.955   0.9507  0.9529
punc         1270      1270    1270  1       1       1
pron          458       473     476  0.9683  0.9622  0.9652
conj          378       426     411  0.8873  0.9197  0.9032
adj           306       349     366  0.8768  0.8361  0.8559
adv           219       245     262  0.8939  0.8359  0.8639
num           153       155     155  0.9871  0.9871  0.9871
tn            136       140     142  0.9714  0.9577  0.9645
fw             85        88      87  0.9659  0.977   0.9714
int            24        25      25  0.96    0.96    0.96
abb        

12. error analysis လုပ်ခြင်း


In [12]:
dev_pred = best[5].predict(X_dv_r)
err = [(w, g, h)
       for sent, gs, hs in zip(dev_sents, y_dv, dev_pred)
       for (w, _), g, h in zip(sent, gs, hs) if g != h]
vocab = {w for s in train_sents for w, _ in s}
oov = sum(1 for w, _, _ in err if w not in vocab)
print(f"DEV mistakes: {len(err)}   of which OOV words: {oov} ({oov/max(len(err),1):.1%})")
print("top confusions:")
for (g, h), n in Counter((g, h) for _, g, h in err).most_common(10):
    print(f"  {g:>5} -> {h:<5} {n:4d}")

print("\n# like: crfsuite tag -r   (left=ref, right=hyp) on first test sentence")
hyp = final.predict([X_te[0]])[0]
for (w, g), h in zip(gold_test[0], hyp):
    flag = " " if g == h else "*"
    print(f" {flag} {w}/{g}\t{h}")


DEV mistakes: 850   of which OOV words: 94 (11.1%)
top confusions:
      v -> part    68
      v -> n       64
   part -> v       49
    adv -> n       48
    adj -> v       47
      v -> adj     43
    adj -> n       41
      n -> v       39
      n -> part    34
   part -> ppm     34

# like: crfsuite tag -r   (left=ref, right=hyp) on first test sentence
   တစ်/tn	tn
   ကိုက်/n	n
   ကို/ppm	ppm
   ဝမ်/n	n
 * ခုနှစ်ထောင်/tn	v
   ပါ/part	part
   ။/punc	punc


## Conclusion

CRF tutorial pipeline ကို POS tagging အတွက် သုံးပြီး baseline template, improved feature, dev tuning, နဲ့ `otest.1k` final evaluation လုပ်ထားပါတယ်။


## Final Result (အကျဉ်းချုပ်)

### Official test (`otest.1k`) — improved model
- Item accuracy: **13007 / 13468 = 0.9658**
- Macro F1: **0.9537**
- Sentence accuracy: **690 / 1000 = 0.6900**

### DEV set error analysis
- DEV mistakes: **850**
- OOV error: **94 = 11.1%**

## Feature template ပြောင်းလိုက်လို့ ဘာတွေကောင်းလာလဲ?

Base (tutorial feature) နဲ့ Improved (script/affix) ကို DEV ပေါ်မှာ နှိုင်းထားပါတယ်။

| Metric | Base · DEV | Improved · DEV | တိုးလာ |
|---|---:|---:|---:|
| Item accuracy | **0.9614** | **0.9661** | **+0.0047** |
| Macro-F1 | **0.9195** | **0.9525** | **+0.0330** |
| Sentence accuracy | **0.6675** | **0.6875** | **+0.0200** |

- **fw** F1: 0.7004 → 0.9767 (**+0.2763**)
- **abb** F1: 0.8667 → 0.9697 (**+0.1030**)
- **num** F1: 0.9496 → 0.9918 (**+0.0422**)

Script / affix feature ထည့်လိုက်လို့ OOV / foreign / number tag တွေ ပိုမှန်လာပြီး Macro-F1 အဓိက တက်လာကြောင်းတွေ့ရှိခဲ့ပါတယ်
